# 🎨 FLUX.1-schnell & SDXL Studio — Colab GPU Backend

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vineshgoyal/colab-flux-schnell/blob/main/notebooks/colab_flux_schnell.ipynb)

> **Important**: To permanently save this notebook in your Google Account so you do not have to recreate it on demand:
> Click **File** ➔ **Save a copy in Drive**.
> Also ensure your runtime is configured for GPU (**Runtime** ➔ **Change runtime type** ➔ **T4 GPU**).


## Step 1: Verify Hardware & GPU Acceleration
Verify that an NVIDIA GPU (Tesla T4, V100, or A100) is allocated with CUDA enabled.

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Warning: No GPU detected! Please change runtime to T4 GPU (Runtime > Change runtime type).")

## Step 2: Mount Google Drive for Persistent Model Cache
Mounting Google Drive ensures model weights (e.g., FLUX.1-schnell, SDXL) and job outputs are saved permanently on Drive, preventing re-downloading on every runtime restart.

In [ ]:
import os
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception as e:
    print("Drive mount note:", e)

DRIVE_DIR = "/content/drive/MyDrive/ai_image_studio_cache"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["STUDIO_DRIVE_CACHE"] = DRIVE_DIR
os.environ["HF_HOME"] = os.path.join(DRIVE_DIR, "huggingface")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"✓ Persistent Drive cache configured at: {DRIVE_DIR}")

## Step 3: Clone or Pull Saved Code from GitHub
Instead of recreating files on demand with fragile cell scripts, this pulls the latest tested code directly from GitHub.

In [ ]:
import os, sys

REPO_URL = "https://github.com/vineshgoyal/colab-flux-schnell.git"
APP_DIR = "/content/colab-flux-schnell"

if not os.path.exists(APP_DIR):
    !git clone {REPO_URL} {APP_DIR}
else:
    !cd {APP_DIR} && git pull

os.chdir(APP_DIR)
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

print(f"✓ Application code ready in {APP_DIR}")

## Step 4: Install Dependencies
Installs PyTorch, Hugging Face Diffusers, FastAPI, and BitsAndBytes.

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r backend/requirements.txt
!pip install -q fastapi uvicorn[standard] python-multipart pydantic pillow
print("✓ Environment dependencies installed.")

## Step 5: Start FastAPI Server & Authenticated Cloudflare Tunnel
Launches the asynchronous GPU inference server and Cloudflare HTTPS tunnel.
Copy the resulting  to your local  file.

In [ ]:
import os, time, re, subprocess

STUDIO_API_TOKEN = os.getenv("STUDIO_API_TOKEN") or "vinesh123"
os.environ["STUDIO_API_TOKEN"] = STUDIO_API_TOKEN
os.environ["STUDIO_AUTH_TOKEN"] = STUDIO_API_TOKEN
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clean existing processes
!pkill -9 -f uvicorn || true
!pkill -9 -f cloudflared || true
time.sleep(2)

# Download cloudflared binary if needed
if not os.path.exists("/content/cloudflared-linux-amd64"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64
    !chmod +x /content/cloudflared-linux-amd64

# Launch Uvicorn in background
server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    ["uvicorn", "backend.server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=APP_DIR,
    env=dict(os.environ, PYTHONPATH=f"{APP_DIR}:" + os.environ.get("PYTHONPATH", "")),
    stdout=server_log, stderr=subprocess.STDOUT
)
time.sleep(3)

# Launch Cloudflare Tunnel
tunnel_log = open("/content/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["/content/cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)

tunnel_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

print("
" + "="*70)
print("🚀 AI Image Studio Backend is Live on Tesla T4 GPU!")
print("="*70)
print(f"Public Tunnel URL: {tunnel_url or 'Failed to get tunnel'}")
print(f"Studio API Token : {STUDIO_API_TOKEN}")
print("="*70)
print("
📋 Update your local .env file:")
print(f"STUDIO_API_URL={tunnel_url or 'http://localhost:8000'}")
print(f"STUDIO_API_TOKEN={STUDIO_API_TOKEN}")
print("="*70)

## Step 6: Backend Health & Status
Verify the local backend is reporting healthy GPU acceleration and VRAM status.

In [ ]:
import urllib.request, json
req = urllib.request.Request(
    "http://127.0.0.1:8000/api/health",
    headers={"Authorization": f"Bearer {STUDIO_API_TOKEN}"}
)
try:
    with urllib.request.urlopen(req, timeout=5) as resp:
        status = json.loads(resp.read().decode())
        print("
✓ GPU Backend Health Status:")
        for k, v in status.items():
            print(f"  {k:20s}: {v}")
except Exception as e:
    print("Health check failed:", e)
    !tail -n 25 /content/server.log